# Evaluating Natural Language Explanations of Food Recommendation System XFoodRec Across Several LLMs

Authors: Junxuan Ling, Tianyi Liu, Anqi Zhou

# 0. Document Instructions

## 0.1. Git Version Control

After making modifications, please use Colab's function of "Save in Github to keep changes" (Github authorization required) to commit the changes to the repository (Your own branch if needed).

**NOTE**: When A modifies this notebook, but have not commited to Github, if B opens this notebook, they will see the Github old version.

## 0.2. Handle persistent data storage

Mount Google Drive instead of relying on Colab's temporary run-time storage.

**IMPORTANT:** Although Google Colab allows collaboration of the notebook, data and folders are originally in Junxuan's drive, this could cause issues when refering to the mounted directory path. To allow the code to work for all of us, (who's not junxuan) please create a **short-cut** of Junxuan's shared folder in our own Google Drive, path `MyDrive/`.

In [1]:
# load from drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0.3 Setup and data loading

Run the notebook from top to bottom after Step 1 has saved the processed files. The sequence is: **load data → build matching resources → parse explanations → extract three claim types → combine claims → verify in 2.2**.

Function-definition cells prepare tools; cells labelled **Build**, **Prepare** or **Extract** apply them; **Inspect** cells summarize the results. Changing an upstream rule requires rerunning its dependent cells.

In [2]:
%pip install -q spacy pyarrow
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.5 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import spacy

project_dir = Path("/content/drive/MyDrive/XFoodRec_Project_Data")
data_dir = project_dir / "raw"
processed_dir = project_dir / "processed"
persona_path = data_dir / "personas.json"
recommendation_path = data_dir / "recommendations.json"
recipe_path = data_dir / "recipes.parquet"
nutrition_fields_path = processed_dir / "nutrition_fields.json"

In [4]:
with persona_path.open("r", encoding="utf-8") as f:
    personas = json.load(f)
with recommendation_path.open("r", encoding="utf-8") as f:
    recommendations = json.load(f)

personas_df = pd.json_normalize(personas)
recipes = pd.read_parquet(recipe_path)
recipes["recipe_id"] = recipes["recipe_id"].astype(str)

In [5]:
explanation_lemmas_df = pd.read_parquet(processed_dir / "explanation_lemmas.parquet")
recipe_lemmas_df = pd.read_parquet(processed_dir / "recipe_lemmas.parquet")
persona_lemmas_df = pd.read_parquet(processed_dir / "persona_lemmas.parquet")
with nutrition_fields_path.open("r", encoding="utf-8") as f:
    nutrition_fields = json.load(f)

print("Explanations:", len(explanation_lemmas_df))
print("Source recipes:", len(recipes))
print("Personas:", len(personas_df))

Explanations: 536
Source recipes: 702
Personas: 40


# 2. Entity Extraction, Fact Verification & Hallucination Detection

## 2.0 Matching helpers reused from Step 1

In [6]:
import re

import spacy

MIN_PREFIX_LENGTH = 4  # word forms sharing a beginning match, e.g. "sugar" / "sugary"

text_nlp = spacy.load("en_core_web_sm", disable=["ner"])

# spaCy adds every processed string to its vocabulary, so keep the model's own word list as loaded.
SPACY_MODEL_WORDS = set(text_nlp.vocab.strings)
_base_forms = {}

In [7]:
def to_list(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    return [str(v).lower() for v in value] if isinstance(value, (list, tuple)) else []


def base_forms(tokens):
    """spaCy lemma of each word on its own, so 'oats'/'oat' and 'potatoes'/'potato' align."""
    new = sorted({t for t in tokens if t not in _base_forms})
    for word, doc in zip(new, text_nlp.pipe(new)):
        _base_forms[word] = doc[0].lemma_.lower() if len(doc) else word
    return [_base_forms[t] for t in tokens]


def term_matches(token, term):
    shorter, longer = sorted((token, term), key=len)
    return token == term or (len(shorter) >= MIN_PREFIX_LENGTH and longer.startswith(shorter))


def contains_sequence(tokens, sequence):
    n = len(sequence)
    return n > 0 and any(tokens[i:i + n] == sequence for i in range(len(tokens) - n + 1))

In [8]:
def noun_phrase_core(chunk, is_name=False):
    """Main noun of a spaCy noun phrase with the words that name it: compounds and -ing words
    ('olive oil', 'baking mix'); describing words are dropped ('frozen vegetables' -> 'vegetable').
    For ingredient names (is_name=True) an -ing word right before the phrase also belongs to it."""
    root = chunk.root
    words = [t for t in chunk if t.i == root.i or (t.head.i == root.i and (t.dep_ == "compound" or t.tag_ == "VBG"))]
    before, after = chunk.start - 1, chunk.end
    if is_name and before >= 0 and chunk.doc[before].tag_ == "VBG" and chunk.doc[before] not in words:
        words = [chunk.doc[before]] + words
    if is_name and after < len(chunk.doc) and chunk.doc[after].tag_ == "VBG" and chunk.doc[after].head.i == root.i:
        words = words + [chunk.doc[after]]  # "salad dressing" when spaCy reads "dressing" as a verb
    terms = base_forms([t.text.lower() for t in words])
    return [] if all(t in text_nlp.Defaults.stop_words for t in terms) else terms


def clean_ingredient_line(line):
    """Drop notes in brackets, including a note that started on the previous line ('in this recipe)')."""
    line = str(line).lower()
    if ")" in line and ("(" not in line or line.index(")") < line.index("(")):
        line = line[line.index(")") + 1:]
    return re.sub(r"\([^)]*\)?", " ", line).strip()


def ingredient_line_names(doc):
    """Ingredient names in one parsed ingredient line: its main noun phrase and phrases joined to it by
    'and' / 'or'. A line that continues the previous one (starts with 'and' / 'or') gives nothing, and a
    'noun' that takes an object ('use salted butter') is read as a verb."""
    if len(doc) and doc[0].pos_ == "CCONJ":
        return []
    chunks = [c for c in doc.noun_chunks if not any(child.dep_ == "dobj" for child in c.root.children)]
    if not chunks:  # a bare name that spaCy does not parse as a noun phrase, e.g. "tahini"
        terms = base_forms([t.text.lower() for t in doc if not (t.is_punct or t.is_space or t.like_num)])
        return [terms] if terms and not all(t in text_nlp.Defaults.stop_words for t in terms) else []
    main = [c for c in chunks
            if c.root.dep_ == "ROOT" or (c.root.head.dep_ == "ROOT" and c.root.dep_ in ("dobj", "compound"))]
    main = main or chunks[:1]
    for c in chunks:
        if c not in main and c.root.dep_ == "conj" and any(c.root.head.i == m.root.i for m in main):
            main.append(c)
    return [name for name in (noun_phrase_core(c, is_name=True) for c in main) if name]

Assemble matching resources

In [9]:
persona_lemmas_df.head()

,document_id,recommendation_id,persona_id,recipe_id,rank,source_field,item_index,persona_text_lemmas
0,user_01:persona_description:0,None,user_01,None,None,persona_description,0,"[vegan, competitive, bodybuilder, focus, lean,..."
1,user_01:dietary_goal:0,None,user_01,None,None,dietary_goal,0,"[muscle, gain]"
2,user_01:dietary_restriction:0,None,user_01,None,None,dietary_restriction,0,[vegan]
3,user_01:dietary_restriction_other:0,None,user_01,None,None,dietary_restriction_other,0,"[high, protein, prefer, low, soy, option]"
4,user_01:liked_ingredient:0,None,user_01,None,None,liked_ingredient,0,[seitan]


In [10]:
recipe_lemmas_df.head()

,document_id,recommendation_id,persona_id,recipe_id,rank,source_field,item_index,recipe_text_lemmas
0,113861:recipe_title:0,None,None,113861,None,recipe_title,0,"[cube, steak, gravy]"
1,113861:recipe_ingredient:0,None,None,113861,None,recipe_ingredient,0,"[cub, steak]"
2,113861:recipe_ingredient:1,None,None,113861,None,recipe_ingredient,1,[oil]
3,113861:recipe_ingredient:2,None,None,113861,None,recipe_ingredient,2,[butter]
4,113861:recipe_ingredient:3,None,None,113861,None,recipe_ingredient,3,[flour]


In [11]:
recipes.head()

,recipe_id,title,recipe_url,image_url,ingredients,ingredients_title,tags,calories_per_serving [cal],calories_per_100g [cal],caloriesfromfat_per_serving [cal],...,dietaryfiber_per_serving [g],dietaryfiber_per_100g [g],sugars_per_serving [g],sugars_per_100g [g],protein_per_serving [g],protein_per_100g [g],who_score,fsa_score,nutri_score,health_score
0,113861,My Own Cube Steak and Gravy,https://www.food.com/recipe/my-own-cube-steak-...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('cubed steak (4-6 pieces)', '2 time(s)...","[cubed steak (4-6 pieces), oil, butter, flour,...","[Moderate Calorie, Vegetarian]",500.5,175.6,216,...,0.8,0.3,0.8,0.3,48.0,16.8,0.266834,0.250,0.00,3.6
1,111212,Our Secret Sirloin Steak,https://www.food.com/recipe/our-secret-sirloin...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('Lea & Perrins Worcestershire Sauce, pl...","[Lea & Perrins Worcestershire Sauce, Lea & Per...","[Moderate Calorie, Vegetarian]",823.6,320.5,502,...,0.4,0.2,3.4,1.3,69.5,27.0,0.186863,0.250,0.00,2.9
2,497021,Mile-High Cabbage Pie #5FIX,https://www.food.com/recipe/mile-high-cabbage-...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('lean ground beef', '1 1/2 time(s) lbs ...","[lean beef, coleslaw mix (cabbage and carrots)...",[Moderate Calorie],803.0,221.2,298,...,7.0,1.9,10.7,2.9,48.7,13.4,0.266693,0.250,0.00,3.6
3,31184,15 Minute Shrimp Scampi,https://www.food.com/recipe/15-minute-shrimp-s...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('extra virgin olive oil', '0.5 time(s) ...","[extra virgin olive oil, onion, cloves garlic,...",[Moderate Calorie],596.2,342.6,416,...,0.7,0.4,2.2,1.3,26.7,15.3,0.169814,0.250,0.00,2.7
4,71739,Easy Garlic Rice,https://www.food.com/recipe/easy-garlic-rice-7...,https://img.sndimg.com/food/image/upload/f_aut...,"{'': [('olive oil', '2 time(s) tablespoons ')...","[olive oil, onion, garlic cloves, long grain r...",[Moderate Calorie],264.2,190.1,70,...,1.3,0.9,1.6,1.2,6.3,4.5,0.222165,0.625,0.25,5.2


In [12]:
def item_key(lemmas):
    # this works (keep only the first noun) because for records under type `liked_ingredients`, 
    # one record corresponds to only 1 ingredient.
    """Core noun phrase of a persona item: 'oats certify gf' -> 'oat', 'frozen vegetable' -> 'vegetable'."""
    doc = text_nlp(" ".join(lemmas))
    chunks = list(doc.noun_chunks)
    if not chunks:
        return base_forms(lemmas)
    root = chunks[0].root
    words = [t.text for t in chunks[0] if t.i == root.i or (t.dep_ == "compound" and t.head.i == root.i)]
    return base_forms(words)


def nutrient_terms_from_data(recipe_columns, nutrition_fields):
    """Nutrients recorded in the data: the recipe nutrient columns ('dietaryfiber_per_serving [g]'
    -> 'fiber') and nutrition_fields.json. Compound column names keep their longest ending that is
    an English word in spaCy's vocabulary."""
    terms = {str(f).lower() for f in nutrition_fields}
    for column in recipe_columns:
        name = re.sub(r"\[[^\]]*\]", "", str(column).lower()).strip()
        if "_per_" not in name:
            continue
        word = name.split("_per_")[0]
        endings = [word[i:] for i in range(len(word)) if word[i:] in SPACY_MODEL_WORDS]
        terms.add(max(endings, key=len) if endings else word)
    return sorted(set(base_forms(sorted(terms))))


def build_persona_values(persona_lemmas_df):
    # Persona fields: the taste profile and the dietary needs.
    TASTE_FIELDS = [
        "liked_ingredient",
        "disliked_ingredient",
        "favorite_cuisine",
    ]
    HEALTH_FIELDS = [
        "dietary_goal",
        "dietary_restriction",
        "food_allergy",
        "health_condition",
    ]

    # These two types of records contain one ingredient per record.
    NOUN_PHRASE_FIELDS = [
        "liked_ingredient",
        "disliked_ingredient",
    ]

    # Allergens are foods ("egg", "milk"), so they are not general health labels.
    NOT_A_LABEL_FIELDS = ["food_allergy"]

    # persona_id: [(source_field, term)]
    persona_values = {}

    for row in persona_lemmas_df.to_dict("records"):
        # Only personal taste and dietary information are relevant.
        field = row["source_field"]

        if field not in TASTE_FIELDS + HEALTH_FIELDS:
            continue

        lemmas = to_list(row["persona_text_lemmas"])
        terms = (
            item_key(lemmas)
            if field in NOUN_PHRASE_FIELDS
            else base_forms(lemmas)
        )

        if terms:
            persona_values.setdefault(
                str(row["persona_id"]),
                [],
            ).append((field, terms))  # source field -> term

    # get taste terms
    taste_values = {
        tuple(terms)
        for values in persona_values.values()
        for field, terms in values
        if field in TASTE_FIELDS
    }

    # health labels
    health_label_values = sorted({
        tuple(terms)  # keep terms
        for values in persona_values.values()
        for field, terms in values
        if (
            # is health not allergy
            field in HEALTH_FIELDS
            and field not in NOT_A_LABEL_FIELDS
            and tuple(terms) not in taste_values
        )
    })

    return persona_values, health_label_values


def build_recipe_titles(recipe_lemmas_df):
    # id: title [lemmas]
    return {
        str(row["recipe_id"]): base_forms(
            to_list(row["recipe_text_lemmas"])
        )
        for row in recipe_lemmas_df.to_dict("records")
        if row["source_field"] == "recipe_title"
    }


def build_ingredient_names(recipe_table):
    # recipe -> ingredients
    line_recipes = []
    line_texts = []

    for recipe_id, lines in zip(
        recipe_table["recipe_id"].astype(str),
        recipe_table["ingredients_title"],
    ):
        for line in to_list(lines):
            line_recipes.append(recipe_id)
            # drop note in brackets and append
            line_texts.append(clean_ingredient_line(line))

    # recipe_id: {(ingredient), (ingredient)}
    ingredient_names = {}

    for recipe_id, doc in zip(
        line_recipes,
        # spaCy tokenize and POS, output doc
        text_nlp.pipe(line_texts, batch_size=256),
    ):
        # all the ingredients retrieved
        for name in ingredient_line_names(doc):
            ingredient_names.setdefault(
                recipe_id,
                set(),
            ).add(tuple(name))

    # construct food term set based on recipe ingredients
    food_terms = set().union(*ingredient_names.values())

    return ingredient_names, food_terms


def build_matching_resources(
    persona_lemmas_df,
    recipe_lemmas_df,
    recipe_table,
    nutrition_fields,
):
    persona_values, health_label_values = build_persona_values(
        persona_lemmas_df
    )

    titles = build_recipe_titles(recipe_lemmas_df)

    ingredient_names, food_terms = build_ingredient_names(
        recipe_table
    )

    return {
        # persona_id: mentioned terms from different sources
        "persona_values": persona_values,

        # health related choices (e.g. diet)
        "health_label_values": [
            list(value)
            for value in health_label_values
        ],

        # recipe_id -> recipe_title
        "titles": titles,

        # all the nutrients ["calorie", "fat", ...]
        "nutrient_terms": nutrient_terms_from_data(
            list(recipe_table.columns),
            nutrition_fields,
        ),

        # recipe_id -> ingredients
        "ingredient_names": ingredient_names,

        # all the ingredients the recipe mentioned
        "food_terms": food_terms,
    }

In [13]:
# preprocess the former lemma dfs, for future analysis

recipe_table = recipes
matching_resources = build_matching_resources(persona_lemmas_df, recipe_lemmas_df, recipe_table, nutrition_fields)

print(
    "Nutrients:",
    matching_resources["nutrient_terms"][:10],
)

print(
    "Health labels:",
    matching_resources["health_label_values"][:10],
)

print(
    "Persona examples:",
    list(matching_resources["persona_values"].items())[:3],
)

print(
    "Recipe ingredient examples:",
    list(matching_resources["ingredient_names"].items())[:3],
)

print(
    "Food terms:",
    list(matching_resources["food_terms"])[:10],
)

Nutrients: ['calorie', 'carb', 'carbohydrate', 'cholesterol', 'fat', 'fiber', 'protein', 'sodium', 'sugar']
Health labels: [['adhd'], ['anemia'], ['anti', 'inflammatory'], ['arfid'], ['asthma'], ['autism', 'spectrum'], ['celiac', 'disease'], ['copd'], ['dairy', 'free'], ['diabete', 'type-1']]
Persona examples: [('user_01', [('dietary_goal', ['muscle', 'gain']), ('dietary_restriction', ['vegan']), ('liked_ingredient', ['seitan']), ('liked_ingredient', ['lentil']), ('liked_ingredient', ['chickpea']), ('liked_ingredient', ['hemp', 'seed']), ('disliked_ingredient', ['mushroom']), ('disliked_ingredient', ['eggplant']), ('favorite_cuisine', ['middle', 'eastern']), ('favorite_cuisine', ['indian']), ('favorite_cuisine', ['mediterranean'])]), ('user_02', [('dietary_goal', ['medical', 'management']), ('dietary_restriction', ['low', 'carb']), ('food_allergy', ['tree', 'nut']), ('health_condition', ['diabete', 'type-1']), ('liked_ingredient', ['chicken']), ('liked_ingredient', ['greek', 'yogurt'])

## 2.1. Extract three types of claims


Prepare original explanations

Create `claim_texts`, `claim_index` and `claim_docs`. Each explanation is parsed once and reused by all three branches.

In [14]:
# Parse every explanation once; 2.1 a-c read their claims from these parses.

# Same ids as the processed lemma table: "<persona id>_rank_<rank>".
claim_texts = {
    f"{entry['persona']['id']}_rank_{rank:02d}": rec["explanation"]
    for entry in recommendations
    for rank, rec in enumerate(entry["recommendations"], start=1)
}
assert set(claim_texts) == set(explanation_lemmas_df["recommendation_id"]), "ids differ from the lemma table"

claim_index = explanation_lemmas_df[["recommendation_id", "persona_id", "recipe_id", "rank"]].copy()
claim_ids = claim_index["recommendation_id"].tolist()
claim_docs = dict(zip(claim_ids, text_nlp.pipe([claim_texts[i] for i in claim_ids], batch_size=64)))

Locate recipe titles

Record character spans of exact title occurrences so title-only mentions can be excluded.

In [15]:
# Where the recipe title appears in each explanation (character spans), so claims inside it are skipped.
title_by_recipe = dict(zip(recipe_table["recipe_id"].astype(str), recipe_table["title"].astype(str).str.lower()))
claim_title_spans = {}
for row in claim_index.itertuples():
    title = title_by_recipe.get(str(row.recipe_id), "")
    text = claim_texts[row.recommendation_id].lower()
    claim_title_spans[row.recommendation_id] = [(m.start(), m.end()) for m in re.finditer(re.escape(title), text)] if title else []

Shared claim context

Helpers identify title overlap, sentence number and negation. These are extraction rules, not truth checks.

In [16]:
def outside_title(span, recommendation_id):
    return not any(span.start_char < end and span.end_char > start for start, end in claim_title_spans[recommendation_id])


def sentence_number(span):
    return next(k for k, sentence in enumerate(span.doc.sents) if sentence.start <= span.start < sentence.end)


def is_negated(token):
    """spaCy's negation mark ('not', 'never') on the word or the words it hangs from, up to its verb,
    or the word being the object of 'without' ('without added sugars'). A 'not' with words of its own
    ('not only ... but also') does not negate."""
    chain = [token]
    while chain[-1].pos_ not in ("VERB", "AUX") and chain[-1].head.i != chain[-1].i:
        chain.append(chain[-1].head)
    return (any(child.dep_ == "neg" and not list(child.children) for t in chain for child in t.children)
            or any(t.dep_ == "prep" and t.lower_ == "without" for t in chain))


print("Explanations parsed:", len(claim_docs))
print("Explanations whose text contains the recipe title:", sum(bool(s) for s in claim_title_spans.values()))

Explanations parsed: 536
Explanations whose text contains the recipe title: 503


### a. Ingredient mentions

Extract ingredient mentions

Recognize food noun phrases outside the recipe title and collect them in `ingredient_claims_df`.

In [17]:
food_terms = matching_resources["food_terms"]

ingredient_claim_rows = []
for recommendation_id, doc in claim_docs.items():
    for chunk in doc.noun_chunks:
        if not outside_title(chunk, recommendation_id):
            continue
        # In "a mix of flavors" or "a variety of vegetables" the thing named comes after "of".
        if any(child.dep_ == "prep" and child.lower_ == "of" for child in chunk.root.children):
            continue
        core = noun_phrase_core(chunk)
        root = base_forms([chunk.root.text.lower()])
        if tuple(core) in food_terms:
            term = core
        elif tuple(root) in food_terms and root[0] not in text_nlp.Defaults.stop_words:
            term = root
        else:
            continue
        ingredient_claim_rows.append({
            "recommendation_id": recommendation_id,
            "claim_type": "ingredient",
            "sentence": sentence_number(chunk),
            "text": chunk.text,
            "term": " ".join(term),
            "negated": is_negated(chunk.root),
            "start_char": chunk.start_char,
            "end_char": chunk.end_char,
        })
ingredient_claims_df = pd.DataFrame(ingredient_claim_rows)

Inspect ingredient extraction

Count mentions and show the most frequent extracted ingredients.

In [18]:
mentions_per_explanation = (
    ingredient_claims_df.groupby("recommendation_id").size()
    .reindex(claim_index["recommendation_id"], fill_value=0)
)
print(f"Ingredient mentions: {len(ingredient_claims_df)} in "
      f"{(mentions_per_explanation > 0).sum()} of {len(mentions_per_explanation)} explanations "
      f"({100 * (mentions_per_explanation > 0).mean():.1f}%)")
display(mentions_per_explanation.value_counts().sort_index().rename_axis("mentions in one explanation")
        .rename("explanations").to_frame())

top_ingredients = (
    ingredient_claims_df.groupby("term")
    .agg(mentions=("recommendation_id", "size"), explanations=("recommendation_id", "nunique"))
    .sort_values("mentions", ascending=False)
)
display(top_ingredients.head(15))

Ingredient mentions: 504 in 333 of 536 explanations (62.1%)


,explanations
mentions in one explanation,
0,203
1,188
2,123
3,19
4,2
5,1


,mentions,explanations
term,,
chicken,96,94
vegetable,63,63
rice,46,45
bean,28,26
banana,19,19
egg,16,16
spinach,15,13
salmon,15,14
chicken breast,11,11


### b. Nutrient claims

Each claim keeps what the sentence says about it:

| Column | What it holds | Example |
|---|---|---|
| `number`, `unit` | An amount given for the nutrient | "33.1g of protein" → 33.1, g; "161 calories" → 161, calorie; "low in calories at 159.6" → 159.6 |
| `qualifier` | The describing word tied to the nutrient in the sentence | "high in protein" → high; "a good source of fiber" → good; "low-calorie" → low; "protein-packed" → packed |

A nutrient word with neither ("combines protein and fiber") is a plain mention. Nutrients joined by "and" share their wording: in "rich in protein and fiber" both are "rich". A number that the sentence structure does not tie to a nutrient belongs to the closest nutrient before it in the same sentence, if that nutrient has no number yet.

Nutrient words said about the user rather than the dish, i.e. in a phrase owned by "your" ("your low-carb diet", "your favorite proteins", "complements your protein-rich meals"), are not nutrient claims. When such a phrase names a user label ("low carb", "high protein") it is a label claim in c.

Recognize nutrients and user references

Identify nutrient words and check whether a phrase describes the user rather than the dish.

In [19]:
nutrient_terms = matching_resources["nutrient_terms"]


def nutrient_name(word):
    """The recorded nutrient a word names ('carbs' -> 'carb', 'sugary' -> 'sugar'), or None."""
    base = base_forms([word.lower()])[0]
    names = [n for n in nutrient_terms if term_matches(base, n)]
    return base if base in names else next(iter(names), None)


def about_user(token):
    """The word sits in a phrase owned by the user ('your low-carb diet', 'your favorite proteins'):
    a second-person possessive on it or on the words it hangs from, up to its verb."""
    while True:
        if any(child.dep_ == "poss" and "2" in child.morph.get("Person") for child in token.children):
            return True
        if token.head.i == token.i or token.pos_ in ("VERB", "AUX"):
            return False
        token = token.head




Recognize amounts and descriptive words

Small grammatical helpers used to locate numerical amounts and words such as high, low and rich.

In [20]:
def is_describing(token):
    """An adjective or past participle: 'high', 'lower', 'rich', 'packed'."""
    return token.tag_ in ("JJ", "JJR", "JJS", "VBN")


def is_amount(token):
    return bool(re.fullmatch(r"\d[\d,]*(\.\d+)?", token.text))


def words_in_front(noun):
    """Describing words placed before a noun: 'low' in 'low-calorie', 'good' in 'a good source'."""
    return [c for c in noun.children if c.dep_ == "amod" and c.i < noun.i and is_describing(c)]


def describing_word_of(prep):
    """The describing word a phrase like 'in calories' belongs to ('high in protein', 'packed with protein'):
    the closest one before it, also when spaCy hangs the phrase on the verb ('is low | in calories')."""
    governor = prep.head
    candidates = [governor] if is_describing(governor) else []
    candidates += [c for c in governor.children if c.dep_ in ("acomp", "attr", "conj") and is_describing(c)]
    return sorted((c for c in candidates if c.i < prep.i), key=lambda c: c.i)[-1:]

Read one nutrient claim

`read_nutrient_claim` links a nutrient to its descriptive words, amount and unit using the sentence parse.

In [21]:
def read_nutrient_claim(token):
    """Describing words and amount (number, unit) of one nutrient word, from the spaCy parse."""
    qualifier, number, unit = words_in_front(token), None, None
    amounts = [c for c in token.children if c.dep_ == "nummod" and c.i < token.i and is_amount(c)]
    if amounts:  # "161 calories", "25g protein"
        number = amounts[0]
        units = [c for c in token.children if c.dep_ == "compound" and number.i < c.i < token.i]
        unit = units[0] if units else token
    # A nutrient joined by "and" to another one shares its wording: "rich in protein and fiber".
    anchor = token
    while anchor.dep_ == "conj" and anchor.head.is_alpha and nutrient_name(anchor.head.text):
        anchor = anchor.head
    if anchor is not token and not qualifier:  # "lower calories and fat"
        qualifier = words_in_front(anchor)
    holder = anchor.head
    if anchor.dep_ == "pobj" and holder.dep_ == "prep":
        qualifier += describing_word_of(holder)
        noun = holder.head
        if holder.lower_ == "of" and noun.pos_ in ("NOUN", "PROPN"):  # "a good source of protein", "21.4g of protein"
            qualifier += [] if qualifier else words_in_front(noun)
            counts = [c for c in noun.children if c.dep_ == "nummod" and is_amount(c)]
            if counts and number is None and anchor is token:
                number, unit = counts[0], noun
    elif anchor.dep_ in ("compound", "amod", "nmod", "npadvmod") and holder.i != anchor.i:
        if is_describing(holder):  # "protein-rich", "protein-packed"
            qualifier.append(holder)
        elif not qualifier:  # "a high protein content", "a moderate calorie option"
            qualifier += [w for w in words_in_front(holder) if w.i < anchor.i]
    elif anchor.dep_ == "dobj":  # "keeping calories low"
        qualifier += [c for c in holder.children if c.dep_ == "oprd" and is_describing(c)]
    return qualifier, number, unit

Extract nutrient claims

Apply the rules to all explanations and create `nutrient_claims_df`. Unattached amounts use the existing nearest-preceding-nutrient rule.

In [22]:
nutrient_claim_rows = []
for recommendation_id, doc in claim_docs.items():
    claims = []
    for token in doc:
        nutrient = token.is_alpha and nutrient_name(token.text)
        if nutrient and outside_title(doc[token.i:token.i + 1], recommendation_id) and not about_user(token):
            qualifier, number, unit = read_nutrient_claim(token)
            claims.append({"token": token, "term": nutrient, "qualifier": qualifier, "number": number, "unit": unit})
    # A number the parse does not tie to a nutrient ("low in calories at 159.6", "protein (28.3g)")
    # belongs to the closest nutrient before it in the same sentence.
    used = {c["number"].i for c in claims if c["number"] is not None}
    for amount in doc:
        if not is_amount(amount) or amount.i in used or not outside_title(doc[amount.i:amount.i + 1], recommendation_id):
            continue
        before = [c for c in claims if c["token"].i < amount.i and c["token"].sent.start == amount.sent.start]
        if before and before[-1]["number"] is None:
            before[-1]["number"] = amount
            before[-1]["unit"] = amount.head if amount.dep_ == "nummod" else None
    for c in claims:
        words = [c["token"]] + c["qualifier"] + [w for w in (c["number"], c["unit"]) if w is not None]
        span = doc[min(w.i for w in words):max(w.i for w in words) + 1]
        nutrient_claim_rows.append({
            "recommendation_id": recommendation_id,
            "claim_type": "nutrient",
            "sentence": sentence_number(span),
            "text": span.text,
            "term": c["term"],
            "qualifier": " ".join(w.lower_ for w in sorted(set(c["qualifier"]), key=lambda w: w.i)) or None,
            "number": float(c["number"].text.replace(",", "")) if c["number"] is not None else None,
            "unit": base_forms([c["unit"].lower_.rstrip(".")])[0] if c["unit"] is not None else None,
            "negated": is_negated(c["token"]),
            "start_char": span.start_char,
            "end_char": span.end_char,
        })
nutrient_claims_df = pd.DataFrame(nutrient_claim_rows)

Inspect nutrient extraction

Summarize numerical claims, qualitative claims, plain mentions and negated claims.

In [23]:
claims_per_explanation = (
    nutrient_claims_df.groupby("recommendation_id").size()
    .reindex(claim_index["recommendation_id"], fill_value=0)
)
print(f"Nutrient claims: {len(nutrient_claims_df)} in "
      f"{(claims_per_explanation > 0).sum()} of {len(claims_per_explanation)} explanations "
      f"({100 * (claims_per_explanation > 0).mean():.1f}%)")

nutrient_forms = pd.DataFrame({
    "claims": [
        nutrient_claims_df["number"].notna().sum(),
        (nutrient_claims_df["number"].isna() & nutrient_claims_df["qualifier"].notna()).sum(),
        (nutrient_claims_df["number"].isna() & nutrient_claims_df["qualifier"].isna()).sum(),
        nutrient_claims_df["negated"].sum(),
    ]
}, index=["with a number", "with a describing word only", "plain mention", "negated (any of the above)"])
display(nutrient_forms)

display(
    nutrient_claims_df.groupby("term")
    .agg(claims=("recommendation_id", "size"), explanations=("recommendation_id", "nunique"),
         with_number=("number", "count"), with_describing_word=("qualifier", "count"))
    .sort_values("claims", ascending=False)
)
display(nutrient_claims_df["qualifier"].value_counts().head(15).rename_axis("describing word").rename("claims").to_frame())

Nutrient claims: 795 in 494 of 536 explanations (92.2%)


,claims
with a number,159
with a describing word only,581
plain mention,55
negated (any of the above),8


,claims,explanations,with_number,with_describing_word
term,,,,
protein,303,291,64,232
calorie,294,274,86,218
carbohydrate,65,65,8,51
fat,45,45,0,43
fiber,43,43,0,35
carb,34,33,1,25
sodium,9,8,0,7
sugar,2,2,0,1


,claims
describing word,
low,130
good,119
rich,70
moderate,58
lower,58
higher,41
high,31
packed,20
healthy,15


### c. Diet / health-label claims

A label claim is a goal, diet or health condition from the user profiles (the health label values of the shared helpers) named in the explanation outside the recipe title, e.g. "your weight loss goal", "a vegetarian option", "your gluten-free needs". A longer form of a label word also counts ("ketogenic" for keto, "energy-boosting" for energy boost); a shorter one does not ("main" is not maintenance). When one label sits inside a longer one ("vegetarian" in "jain vegetarian"), only the longer one is kept.

A label that contains a nutrient word ("low carb", "high protein") counts here only when it is said about the user ("your low-carb diet"); said about the dish ("a high-protein dish") it is a nutrient claim in b. General wording that names no label ("fits your dietary restrictions", "free from allergens") is not a label claim.

---------

Label vocabulary and matching rule

Use the profile-derived labels and distinguish nutrient-related user labels from dish nutrition claims.

In [24]:
label_values = matching_resources["health_label_values"]
# A label with a nutrient word ("low carb") counts here only when said about the user ("your low-carb diet");
# said about the dish ("a low-carb dish") it is a nutrient claim in b.
nutrient_labels = {tuple(value) for value in label_values if any(nutrient_name(word) for word in value)}


def extends(word, part):
    """The explanation word is the label word or a longer form of it ('keto' -> 'ketogenic',
    'boost' -> 'booster'); a shorter word ('main' for 'maintenance') does not count."""
    return word == part or (len(part) >= MIN_PREFIX_LENGTH and word.startswith(part))

Extract label claims

Match labels outside recipe titles, remove contained shorter matches and create `label_claims_df`.

In [25]:
label_claim_rows = []
for recommendation_id, doc in claim_docs.items():
    words = [t for t in doc if not (t.is_punct or t.is_space)]  # "gluten-free" reads as "gluten free"
    bases = base_forms([t.lower_ for t in words])
    found = []
    for value in label_values:
        n = len(value)
        for i in range(len(words) - n + 1):
            if all(extends(base, part) for base, part in zip(bases[i:i + n], value)):
                found.append((doc[words[i].i:words[i + n - 1].i + 1], value))
    for span, value in found:
        # A value inside a longer one ("vegetarian" in "jain vegetarian") is not a claim of its own.
        if any(other.start <= span.start and span.end <= other.end and len(other) > len(span) for other, _ in found):
            continue
        if not outside_title(span, recommendation_id):
            continue
        if tuple(value) in nutrient_labels and not about_user(span.root):
            continue
        label_claim_rows.append({
            "recommendation_id": recommendation_id,
            "claim_type": "label",
            "sentence": sentence_number(span),
            "text": span.text,
            "term": " ".join(value),
            "negated": is_negated(span.root),
            "start_char": span.start_char,
            "end_char": span.end_char,
        })
label_claims_df = pd.DataFrame(label_claim_rows)

Inspect label extraction

Count label mentions, affected explanations and negated mentions.

In [26]:
labels_per_explanation = (
    label_claims_df.groupby("recommendation_id").size()
    .reindex(claim_index["recommendation_id"], fill_value=0)
)
print(f"Diet / health-label claims: {len(label_claims_df)} in "
      f"{(labels_per_explanation > 0).sum()} of {len(labels_per_explanation)} explanations "
      f"({100 * (labels_per_explanation > 0).mean():.1f}%), negated: {label_claims_df['negated'].sum()}")
display(
    label_claims_df.groupby("term")
    .agg(claims=("recommendation_id", "size"), explanations=("recommendation_id", "nunique"))
    .sort_values("claims", ascending=False)
)

Diet / health-label claims: 252 in 231 of 536 explanations (43.1%), negated: 2


,claims,explanations
term,,
weight loss,52,52
muscle gain,45,45
vegetarian,35,35
energy boost,28,28
maintenance,25,25
medical management,13,13
vegan,11,11
high protein,8,8
anti inflammatory,7,7


### All claims

The three tables are stacked into `claims_df`, one row per claim, with the persona, recipe and rank of the explanation. Columns that do not apply to a claim type (`qualifier`, `number`, `unit` outside b) are empty.

----
Combine the three claim tables

Join the extracted rows to their persona, recipe and rank identifiers. `claims_df` is the main input for Step 2.2.

In [27]:
claim_tables = {"ingredient": ingredient_claims_df, "nutrient": nutrient_claims_df, "label": label_claims_df}
claims_df = (
    pd.concat(claim_tables.values(), ignore_index=True)
    .merge(claim_index, on="recommendation_id", how="left", validate="many_to_one")
)
claims_df = claims_df[
    ["recommendation_id", "persona_id", "recipe_id", "rank", "claim_type", "sentence", "text", "term",
     "qualifier", "number", "unit", "negated", "start_char", "end_char"]
]



Extraction summary

Report extracted claims and coverage. These counts do not measure hallucinations.

In [28]:
explanation_count = len(claim_index)
claim_summary = pd.DataFrame({
    "claims": [len(t) for t in claim_tables.values()] + [len(claims_df)],
    "negated": [int(t["negated"].sum()) for t in claim_tables.values()] + [int(claims_df["negated"].sum())],
    "explanations": [t["recommendation_id"].nunique() for t in claim_tables.values()]
                    + [claims_df["recommendation_id"].nunique()],
}, index=list(claim_tables) + ["any"])
claim_summary["percent of explanations"] = (100 * claim_summary["explanations"] / explanation_count).round(1)
display(claim_summary)
print("Explanations without any claim:", explanation_count - claims_df["recommendation_id"].nunique())

,claims,negated,explanations,percent of explanations
ingredient,504,1,333,62.1
nutrient,795,8,494,92.2
label,252,2,231,43.1
any,1551,11,527,98.3


Explanations without any claim: 9


## 2.2 & 2.3 Compare and verify extracted claims with the actual data & detect hallucination

Each claim receives verdict of `supported`, `contradicted` or `unverified`, with the source evidence and rule used.

Missing evidence is not treated as proof of correctness.


prepare claims to be verified, find the original sentence corresponding to each claim.

In [29]:
# prepare the claims to be verified ###################################

verification_claims = claims_df.reset_index(drop=True)
verification_claims.insert(0, "claim_id", range(len(verification_claims)))
recipe_lookup = recipes.set_index("recipe_id").to_dict("index") # recipe to map
profile_lookup = {p["id"]: p["profile"] for p in personas} # profile to map

print("verification_claims")
print(verification_claims)

verification_claims
      claim_id recommendation_id persona_id recipe_id  rank  claim_type  \
0            0   user_01_rank_01    user_01     81968     1  ingredient   
1            1   user_01_rank_01    user_01     81968     1  ingredient   
2            2   user_01_rank_02    user_01     10614     2  ingredient   
3            3   user_01_rank_03    user_01     94673     3  ingredient   
4            4   user_01_rank_04    user_01    364252     4  ingredient   
...        ...               ...        ...       ...   ...         ...   
1546      1546   user_40_rank_09    user_40    119096     9       label   
1547      1547   user_40_rank_10    user_40     30745    10       label   
1548      1548   user_40_rank_11    user_40     14788    11       label   
1549      1549   user_40_rank_12    user_40    119417    12       label   
1550      1550   user_40_rank_15    user_40     17107    15       label   

      sentence           text         term qualifier  number unit  negated  \
0

In [30]:
# preprocess words ###################################
def verification_words(text):
    return base_forms([token.lower_ for token in text_nlp.make_doc(text) if token.is_alpha])

# recipe_id: [(original line 1, preprocessed word list 1), (original line 2, preprocessed word list 2)]
source_ingredients = {
    row.recipe_id: [(line, verification_words(line)) for line in row.ingredients_title]
    for row in recipes.itertuples()
}

print(next(iter(source_ingredients.items()))[0])
print(json.dumps(next(iter(source_ingredients.items()))[1],indent=2))


113861
[
  [
    "cubed steak (4-6 pieces)",
    [
      "cub",
      "steak",
      "piece"
    ]
  ],
  [
    "oil",
    [
      "oil"
    ]
  ],
  [
    "butter",
    [
      "butter"
    ]
  ],
  [
    "flour",
    [
      "flour"
    ]
  ],
  [
    "salt",
    [
      "salt"
    ]
  ],
  [
    "onion powder",
    [
      "onion",
      "powder"
    ]
  ],
  [
    "garlic powder",
    [
      "garlic",
      "powder"
    ]
  ],
  [
    "black pepper",
    [
      "black",
      "pepper"
    ]
  ],
  [
    "cream of mushroom soup",
    [
      "cream",
      "of",
      "mushroom",
      "soup"
    ]
  ],
  [
    "water",
    [
      "water"
    ]
  ],
  [
    "heavy whipping cream",
    [
      "heavy",
      "whip",
      "cream"
    ]
  ]
]


In [31]:
# find the original sentence of the claim and add to the records. ###################################

# rec_id: 'user_01_rank_01' (created during last Step 1)
# sentence list
sentences = {
    rec_id: [sentence.text for sentence in doc.sents]
    for rec_id, doc in claim_docs.items()
}

verification_claims["sentence_text"] = [
    sentences[row.recommendation_id][row.sentence] # rec_id, sentence_id -> sentence text
    for row in verification_claims.itertuples()
]

print(verification_claims)

      claim_id recommendation_id persona_id recipe_id  rank  claim_type  \
0            0   user_01_rank_01    user_01     81968     1  ingredient   
1            1   user_01_rank_01    user_01     81968     1  ingredient   
2            2   user_01_rank_02    user_01     10614     2  ingredient   
3            3   user_01_rank_03    user_01     94673     3  ingredient   
4            4   user_01_rank_04    user_01    364252     4  ingredient   
...        ...               ...        ...       ...   ...         ...   
1546      1546   user_40_rank_09    user_40    119096     9       label   
1547      1547   user_40_rank_10    user_40     30745    10       label   
1548      1548   user_40_rank_11    user_40     14788    11       label   
1549      1549   user_40_rank_12    user_40    119417    12       label   
1550      1550   user_40_rank_15    user_40     17107    15       label   

      sentence           text         term qualifier  number unit  negated  \
0            1      c

helper functions

In [32]:
# to record the verdict
def verdict(status, reason, evidence="", rule=""):
    return {"verification_status": status, "reason": reason, "evidence": evidence, "rule": rule}

# for showing detection results.
def inspect_verification(table):
    display(table["verification_status"].value_counts().to_frame("claims"))
    display(table[["claim_id", "recipe_id", "text", "verification_status", "reason", "evidence"]].head(10))

### a. Ingredients → recipe's ingredient list

Verify ingredient presence

Match the claim term with source ingredient lines.

Absence means “not found in the recorded ingredient list” under this matching rule. Negated mentions reverse the presence check.

Optional ingredients, alternatives and serving suggestions are left for review.

In [33]:
# to match for
"""
pairs / paired / pairing
served with
instead
optional
can add
"""
# for recognizing suggestions, food not included in the recipe but in fact serves as
# suggestions are not hallucination.

CONTEXT_REVIEW = re.compile(
    r"\b(?:pair(?:s|ed|ing)?|serve(?:d)? with|served alongside|substitut\w*|"
    r"swap\w*|replac\w*|instead|optional|could add|can add|can be made|"
    r"can be adapt\w*|can be customiz\w*)\b", re.I
)

In [34]:
print(source_ingredients)

{'113861': [('cubed steak (4-6 pieces)', ['cub', 'steak', 'piece']), ('oil', ['oil']), ('butter', ['butter']), ('flour', ['flour']), ('salt', ['salt']), ('onion powder', ['onion', 'powder']), ('garlic powder', ['garlic', 'powder']), ('black pepper', ['black', 'pepper']), ('cream of mushroom soup', ['cream', 'of', 'mushroom', 'soup']), ('water', ['water']), ('heavy whipping cream', ['heavy', 'whip', 'cream'])], '111212': [('Lea & Perrins Worcestershire Sauce', ['lea', 'perrin', 'worcestershire', 'sauce']), ('Lea & Perrins Worcestershire Sauce', ['lea', 'perrin', 'worcestershire', 'sauce']), ('lemon juice', ['lemon', 'juice']), ('olive oil', ['olive', 'oil']), ('instant onion', ['instant', 'onion']), ('salt', ['salt']), ('instant garlic', ['instant', 'garlic']), ('sirloin steaks', ['sirloin', 'steak']), ('butter', ['butter']), ('parsley', ['parsley'])], '497021': [('lean beef', ['lean', 'beef']), ('coleslaw mix (cabbage and carrots)', ['coleslaw', 'mix', 'cabbage', 'and', 'carrot']), ('S

In [35]:
import nltk
from nltk.corpus import wordnet as wn
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [36]:
import re

def get_wordnet_related_terms(term_tokens):
    """
    Retrieve the original term, its WordNet synonyms, and all hyponyms.
    Uses a safe BFS approach to avoid the known NLTK syn.closure() 'NoneType' bug.
    Returns a set of tuples to maintain compatibility with contains_sequence.
    """
    term_str = "_".join(term_tokens)
    valid_terms = {tuple(term_tokens)}

    # Retrieve noun synsets for the target term
    for syn in wn.synsets(term_str, pos=wn.NOUN):
        # 1. Add synonyms (lemmas of the current synset itself)
        for lemma in syn.lemmas():
            valid_terms.add(tuple(lemma.name().replace('_', ' ').lower().split()))

        # 2. Add hyponyms safely using Breadth-First Search (BFS)
        queue = list(syn.hyponyms())
        visited = set()

        while queue:
            current_syn = queue.pop(0)
            if current_syn not in visited:
                visited.add(current_syn)

                # Add all lemmas of the current hyponym
                for lemma in current_syn.lemmas():
                    valid_terms.add(tuple(lemma.name().replace('_', ' ').lower().split()))

                # Add the next level of hyponyms to the queue
                queue.extend(current_syn.hyponyms())

    return valid_terms


def verify_ingredient(row):
    sentence = row["sentence_text"] # get the raw sentence where the claim is detected

    # Context check: cannot directly verdict if the sentence involves preference, substitution, or serving context
    if CONTEXT_REVIEW.search(sentence) or about_user(
        claim_docs[row["recommendation_id"]].char_span(int(row["start_char"]), int(row["end_char"]), alignment_mode="expand").root
    ): #
        return verdict("unverified", "Preference, substitution or serving context; check the original sentence.", sentence) #

    # matching the claim term with the original sentence ingredient list ##############

    term = verification_words(row["term"]) # claim term
    lines = source_ingredients[str(row["recipe_id"])] # ingredient list of the source recipe

    # expand vocabulary: include WordNet hyponyms/synonyms (e.g. vegetables, cabbages)
    valid_term_sequences = get_wordnet_related_terms(term)

    matches = [
        raw for raw, words in lines
        if any(contains_sequence(words, list(t)) for t in valid_term_sequences)
    ]

    # if the matched ingredients is optional in the recipe, cannot verdict
    if matches and any(re.search(r"\b(?:or|optional|substitute)\b", line, re.I) for line in matches): #
        return verdict("unverified", "Matching source ingredient is optional or an alternative.", "; ".join(matches)) #

    # direct verdict: supported (adjusting for negation)
    present = bool(matches) #
    consistent = not present if row["negated"] else present #

    return verdict(
        "supported" if consistent else "contradicted", #
        "Ingredient or its hyponym found in source." if present else "Ingredient not found in the recorded list; review synonyms before final labeling.",
        "; ".join(matches) if matches else "; ".join(raw for raw, _ in lines), #
        "Normalized phrase or WordNet hyponym match; negation reverses presence."
    )

ingredient_checks = verification_claims.loc[verification_claims["claim_type"].eq("ingredient")].copy()
ingredient_checks = ingredient_checks.join(ingredient_checks.apply(verify_ingredient, axis=1).apply(pd.Series))

Inspect ingredient verification

In [37]:
inspect_verification(ingredient_checks)

,claims
verification_status,
supported,332
unverified,119
contradicted,53


,claim_id,recipe_id,text,verification_status,reason,evidence
0,0,81968,chickpeas,supported,Ingredient or its hyponym found in source.,cooked chickpeas
1,1,81968,spinach,supported,Ingredient or its hyponym found in source.,package frozen spinach
2,2,10614,black beans,unverified,Matching source ingredient is optional or an a...,black beans (or red kidney
3,3,94673,lentils,supported,Ingredient or its hyponym found in source.,red lentils
4,4,364252,The olive oil,supported,Ingredient or its hyponym found in source.,olive oil
5,5,12569,protein-rich black beans,supported,Ingredient or its hyponym found in source.,black beans (and liquid)
6,6,27778,black beans,supported,Ingredient or its hyponym found in source.,black beans
7,7,17107,vegetables,unverified,"Preference, substitution or serving context; c...",This dish can be a hearty addition to your mea...
8,8,27778,black beans,supported,Ingredient or its hyponym found in source.,black beans
9,9,12633,meat,supported,Ingredient or its hyponym found in source.,stew meat


In [38]:
# for human inspect

display(
    ingredient_checks.loc[
        ingredient_checks["verification_status"].eq("contradicted"),
        ["claim_id", "recipe_id", "text", "sentence_text", "reason", "evidence"]
    ]
)

,claim_id,recipe_id,text,sentence_text,reason,evidence
25,25,12569,chicken,The Chicken Rice Casserole is a comforting dis...,Ingredient not found in the recorded list; rev...,black beans (and liquid); onion; cloves garlic...
37,37,14409,strawberries,The combination of spinach and strawberries ma...,Ingredient not found in the recorded list; rev...,spinach (I pinch off the larger stems); slicer...
41,41,17471,chickpeas,It’s packed with protein from chickpeas and is...,Ingredient not found in the recorded list; rev...,vegetable oil; onions (peeled and minced); clo...
44,44,25690,whole wheat flour,Pancakes made with whole wheat flour are a who...,Ingredient not found in the recorded list; rev...,egg; milk; butter; flour; honey; baking powder...
45,45,25690,fruits,Enjoy them topped with fruits for an extra nut...,Ingredient not found in the recorded list; rev...,egg; milk; butter; flour; honey; baking powder...
53,53,34716,Rice,Ramen Fried 'Rice' is a creative take on a ric...,Ingredient not found in the recorded list; rev...,package pork-flavored ramen noodles (or smoked...
56,56,12569,tuna,The combination of beans and tuna provides a g...,Ingredient not found in the recorded list; rev...,black beans (and liquid); onion; cloves garlic...
59,59,31546,Chicken,Chicken & Long Grain Rice is a hearty meal tha...,Ingredient not found in the recorded list; rev...,lettuce; cheddar cheese; beef; package taco se...
60,60,31546,Long Grain Rice,Chicken & Long Grain Rice is a hearty meal tha...,Ingredient not found in the recorded list; rev...,lettuce; cheddar cheese; beef; package taco se...
61,61,34716,Fried Rice,Fried Rice is a classic dish that can be tailo...,Ingredient not found in the recorded list; rev...,package pork-flavored ramen noodles (or smoked...


### b. Nutrient claims → recipe's nutrition fields

Numerical and qualitative rules

Numeric amounts are compared with the source's nutrient. An explicit “per 100 g” uses that column; otherwise one serving is assumed. Values in g/mg are converted.

The project tolerance is **max(0.1 source units, 5% of the source value)**.

Unsupported units, percentages, ranges and negated numeric claims require review.

Qualitative rules use the [European Commission nutrition-claim thresholds](https://food.ec.europa.eu/food-safety/labelling-and-nutrition/nutrition-and-health-claims/nutrition-claims_en):

| Rule | Source-relative threshold |
|---|---|
| High / source of protein | At least 20% / 12% of energy from protein |
| High / source of fibre | At least 6 / 3 g per 100 g, OR 3 / 1.5 g per 100 kcal |
| Low fat / sugars | At most 3 / 5 g per 100 g |
| Low energy / sodium | At most 40 kcal / 120 mg per 100 g |

`rich` and `packed` map to high; `good` and `source` map to source. “Moderate”, “healthy”, “higher/lower” without a comparator, and unsupported nutrient/rule pairs remain unverified.

The dataset's calorie fields labelled `[cal]` are interpreted as dietary kcal.

Check qualitative thresholds

In [39]:
def qualitative_nutrition(recipe, term, qualifier):
    # only do evaluation based on EU qualitative thresholds
    if qualifier not in {"high", "source", "low"}:
        return verdict("unverified", "No absolute threshold for this qualifier.")

    # standardize unit to per-100g mass
    value, column = nutrient_value(recipe, term, "per_100g")
    if not np.isfinite(value):
        return verdict("unverified", "Source nutrient is missing.", column)

    # protein logic: evaluated by energy contribution (protein provides 4 kcal/g)
    # High: >= 20% total energy | Source: >= 12% total energy
    if term == "protein" and qualifier in {"high", "source"}:
        energy = float(recipe["calories_per_100g [cal]"])
        if not np.isfinite(energy) or energy <= 0: # invalid number
            return verdict("unverified", "Positive source energy is needed for the protein energy share.")

        share = 100 * value * 4 / energy
        threshold = 20 if qualifier == "high" else 12
        passed, evidence, rule = share >= threshold, f"Protein energy share: {share:.2f}%", f">= {threshold}% energy from protein"

    # fiber logic: Evaluated by dual criteria (mass basis OR energy basis)
    # High: >= 6g/100g or 3g/100kcal | Source: >= 3g/100g or 1.5g/100kcal
    elif term == "fiber" and qualifier in {"high", "source"}:
        energy = float(recipe["calories_per_100g [cal]"])
        if not np.isfinite(energy) or energy <= 0:
            return verdict("unverified", "Positive source energy is needed for both fibre criteria.")

        threshold = 6 if qualifier == "high" else 3
        per_energy = value * 100 / energy
        passed = value >= threshold or per_energy >= threshold / 2
        evidence = f"Fibre: {value:g} g/100g; {per_energy:.2f} g/100kcal"
        rule = f">= {threshold} g/100g OR >= {threshold / 2:g} g/100kcal"

    # low logic: Evaluated by absolute mass-based limits per 100g
    elif qualifier == "low" and term in {"fat", "sugar", "calorie", "sodium"}:
        threshold = {"fat": 3, "sugar": 5, "calorie": 40, "sodium": 120}[term]
        passed, evidence, rule = value <= threshold, f"{column}: {value:g}", f"<= {threshold} per 100g (study mass-basis rule)"

    else:
        return verdict("unverified", "No documented threshold for this nutrient/qualifier pair.", column)

    # verdict based on whether the calculated value meets the established threshold
    return verdict("supported" if passed else "contradicted", "Compared with the documented threshold.", evidence, rule)

Verify each nutrient claim

Evaluate numeric and qualitative parts independently, then retain both component statuses.

In [40]:
NUTRIENT_FIELDS = {
    "calorie": ("calories", "cal", "kcal"),
    "protein": ("protein", "g", "g"),
    "carb": ("totalcarbohydrate", "g", "g"),
    "carbohydrate": ("totalcarbohydrate", "g", "g"),
    "fat": ("totalfat", "g", "g"),
    "fiber": ("dietaryfiber", "g", "g"),
    "sugar": ("sugars", "g", "g"),
    "sodium": ("sodium", "mg", "mg"),
    "cholesterol": ("cholesterol", "mg", "mg")
}

NUMERIC_REL_TOL = 0.05
NUMERIC_ABS_TOL = 0.1

# according to the basis, return the true value
def nutrient_value(recipe, term, basis):
    stem, column_unit, _ = NUTRIENT_FIELDS[term]
    column = f"{stem}_{basis} [{column_unit}]"
    return float(recipe[column]), column

In [41]:
QUALIFIER_MAP = {"high": "high", "rich": "high", "packed": "high", "good": "source", "source": "source", "low": "low"}

def verify_nutrient(row):
    recipe = recipe_lookup[str(row["recipe_id"])]
    term, sentence = row["term"], row["sentence_text"]

    if term not in NUTRIENT_FIELDS: # not detected
        return verdict("unverified", "No source-column mapping for the extracted nutrient.")
    if CONTEXT_REVIEW.search(sentence): # the explanation suggests a modification
        return verdict("unverified", "Recipe modification or serving context requires review.", sentence)

    results = {}

    # Numeric Claim Verification
    if pd.notna(row["number"]):
        # default assumption is per serving
        basis = "per_100g" if re.search(r"\b(?:per|/|in|for)\s*100\s*g(?:rams?)?\b", sentence, re.I) else "per_serving"
        actual, column = nutrient_value(recipe, term, basis)
        _, _, src_unit = NUTRIENT_FIELDS[term]

        raw_unit = str(row["unit"]).lower() if pd.notna(row["unit"]) else ""
        unit = {"gram": "g", "grams": "g", "milligram": "mg", "milligrams": "mg", "calorie": "kcal", "calories": "kcal"}.get(raw_unit, raw_unit)
        unit = src_unit if unit in {"", term} else unit

        factor = {("g", "mg"): 1000, ("mg", "g"): 0.001}.get((unit, src_unit), 1 if unit == src_unit else None)
        ambiguous = re.search(r"%|percent|per\s*100\s*ml|\bbetween\b|\d\s*[-–]\s*\d|\b(?:less|more) than\b|\bup to\b", sentence, re.I)

        if row["negated"] or ambiguous or factor is None or not np.isfinite(actual):
            results["numeric"] = verdict("unverified", "Negation, range, unsupported unit/basis, or missing source value.", sentence)
        else:
            claimed = float(row["number"]) * factor
            tolerance = max(NUMERIC_ABS_TOL, NUMERIC_REL_TOL * abs(actual))
            passed = abs(claimed - actual) <= tolerance
            results["numeric"] = verdict(
                "supported" if passed else "contradicted",
                "Compared with source nutrition.",
                f"Claim={claimed:g} {src_unit}; {column}={actual:g}; tolerance={tolerance:g}",
                f"Absolute diff <= max(0.1, 5% of source); basis={basis}; inferred unit={unit==src_unit}"
            )

    # Qualitative Claim Verification
    if pd.notna(row["qualifier"]):
        qualifier = str(row["qualifier"]).lower().strip()
        results["qualitative"] = (
            verdict("unverified", "Negated qualitative wording requires a scope check.") if row["negated"]
            else qualitative_nutrition(recipe, term, QUALIFIER_MAP.get(qualifier, qualifier))
        )

    if not results:
        return verdict("unverified", "Plain nutrient mention has no numerical or qualitative claim to check.")

    # Aggregate Final Verdict
    statuses = set(x["verification_status"] for x in results.values())
    status = "contradicted" if "contradicted" in statuses else "unverified" if "unverified" in statuses else "supported"

    result = verdict(
        status,
        " | ".join(f"{k}: {v['reason']}" for k, v in results.items()),
        " | ".join(f"{k}: {v['evidence']}" for k, v in results.items()),
        " | ".join(f"{k}: {v['rule']}" for k, v in results.items())
    )
    result.update({f"{k}_status": v["verification_status"] for k, v in results.items()})

    return result

Verify and inspect nutrient claims

In [42]:
nutrient_checks = verification_claims.loc[verification_claims["claim_type"].eq("nutrient")].copy()
nutrient_checks = nutrient_checks.join(nutrient_checks.apply(verify_nutrient, axis=1).apply(pd.Series))
inspect_verification(nutrient_checks)
display(nutrient_checks[["numeric_status", "qualitative_status"]].value_counts(dropna=False).rename("claims").to_frame())

,claims
verification_status,
unverified,369
supported,311
contradicted,115


,claim_id,recipe_id,text,verification_status,reason,evidence
504,504,81968,packed with protein,contradicted,qualitative: Compared with the documented thre...,qualitative: Protein energy share: 17.23%
505,505,10614,high protein,supported,qualitative: Compared with the documented thre...,qualitative: Protein energy share: 23.02%
506,506,10614,low in fat,supported,qualitative: Compared with the documented thre...,qualitative: totalfat_per_100g [g]: 1.2
507,507,94673,rich in protein,supported,qualitative: Compared with the documented thre...,qualitative: Protein energy share: 20.11%
508,508,94673,rich in protein and fiber,supported,qualitative: Compared with the documented thre...,qualitative: Fibre: 10.6 g/100g; 5.86 g/100kcal
509,509,364252,low in calories,contradicted,qualitative: Compared with the documented thre...,qualitative: calories_per_100g [cal]: 82.9
510,510,364252,healthy fats,unverified,qualitative: No absolute threshold for this qu...,qualitative:
511,511,12569,protein-rich,contradicted,qualitative: Compared with the documented thre...,qualitative: Protein energy share: 8.39%
512,512,12569,carbs,unverified,Plain nutrient mention has no numerical or qua...,
513,513,27778,good amount of protein,supported,qualitative: Compared with the documented thre...,qualitative: Protein energy share: 15.91%


claims
numeric_status qualitative_status        
NaN            unverified             273
               supported              181
supported      NaN                    123
NaN            contradicted           110
               NaN                     74
supported      unverified              20
               supported                7
               contradicted             4
unverified     NaN                      2
contradicted   NaN                      1

### c. Diet/health-label claims → both the recipe's ingredients and the persona's restrictions/allergies/conditions

Ingredient conflict rules

Explicit ingredient groups below provide a reproducible screen for common dietary exclusions and the nine recorded allergy categories. The lists are project matching rules, not a complete allergen database. Alternatives, “free-from” and imitation products require review. Recipe tags are not used as ground truth.

A detected forbidden ingredient contradicts a positive compatibility claim. No detected conflict alone does **not** establish allergen safety, gluten content, religious certification or disease suitability. Unsupported diets and clinical outcomes remain unverified. Full persona fields (including `other`) are retained for review.

In [43]:
FOOD_GROUPS = {
    "red_meat": "beef|pork|bacon|ham|lamb|veal|venison|prosciutto|pancetta|lard|pepperoni",
    "poultry": "chicken|turkey|duck|goose",
    "fish": "fish|salmon|tuna|anchovy|sardine|cod|haddock|trout|tilapia|fish sauce",
    "shellfish": "shrimp|prawn|crab|lobster|clam|mussel|oyster|scallop|crayfish",
    "milk": "milk|cream|butter|cheese|yogurt|yoghurt|whey|casein|ghee|parmesan|cheddar|mozzarella|feta",
    "eggs": "egg|mayonnaise|meringue",
    "peanuts": "peanut|groundnut",
    "tree_nuts": "almond|walnut|cashew|pecan|pistachio|hazelnut|macadamia|brazil nut",
    "sesame": "sesame|tahini",
    "soy": "soy|soya|tofu|tempeh|edamame|miso",
    "wheat": "wheat|semolina|bulgur|couscous|seitan|spelt|farro",
    "gluten_grains": "wheat|semolina|bulgur|couscous|seitan|spelt|farro|barley|rye",
    "apples": "apple|applesauce",
    "gelatin": "gelatin|gelatine",
    "honey": "honey"
}
FOOD_GROUPS = {group: [verification_words(term) for term in terms.split("|")] for group, terms in FOOD_GROUPS.items()}
DIET_GROUPS = {
    "vegan": ["red_meat", "poultry", "fish", "shellfish", "milk", "eggs", "gelatin", "honey"],
    "vegetarian": ["red_meat", "poultry", "fish", "shellfish", "gelatin"],
    "pescatarian": ["red_meat", "poultry"],
    "no red meat": ["red_meat"],
    "dairy free": ["milk"],
    "gluten free": ["gluten_grains"]
}
ALLERGY_GROUPS = {
    "apples": "apples", "eggs": "eggs", "milk": "milk", "peanuts": "peanuts",
    "sesame": "sesame", "shellfish": "shellfish", "soy": "soy", "tree nuts": "tree_nuts", "wheat": "wheat"
}

def label_key(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()



Find explicit ingredient conflicts

In [44]:
def forbidden_ingredients(recipe_id, groups):
    certain, uncertain = [], []
    for raw, words in source_ingredients[str(recipe_id)]:
        matched_groups = [group for group in groups if any(
            contains_sequence(words, term) for term in FOOD_GROUPS[group]
        )]
        if not matched_groups:
            continue
        line = " ".join(words)
        ambiguous = re.search(r"\b(?:or|optional|free|substitute|imitation|vegan|vegetarian|non dairy|nondairy)\b", line)
        # A plant-based dairy alternative is ambiguous for milk, but still evidence for its own allergen.
        plant_dairy = re.search(r"\b(?:coconut|almond|soy|soya|oat|rice|cashew|peanut|apple|cocoa)\b.*\b(?:milk|cream|butter|cheese|yogurt)\b", line)
        if plant_dairy and set(matched_groups) == {"milk"}:
            ambiguous = True
        (uncertain if ambiguous else certain).append(raw)
    return certain, uncertain



Evaluate one persona requirement

Ingredient exclusions use source lines. High-protein, low-fat and low-sodium requirements reuse the nutritional rules above; other requirements remain open for review.

In [45]:
NUTRITION_LABELS = {"high protein": ("protein", "high"), "low fat": ("fat", "low"), "low sodium": ("sodium", "low")}

def check_profile_requirement(recipe_id, field, value):
    key = label_key(value)
    if field == "foodAllergies":
        groups = [ALLERGY_GROUPS[key]] if key in ALLERGY_GROUPS else None
    elif field == "healthConditions":
        groups = ["gluten_grains"] if key == "celiac disease" else None
    else:
        groups = DIET_GROUPS.get(key)
    if groups is not None:
        conflicts, ambiguous = forbidden_ingredients(recipe_id, groups)
        if conflicts:
            return verdict("contradicted", "Explicit source ingredient conflicts with this requirement.", "; ".join(conflicts), f"Ingredient exclusion: {key}")
        return verdict("unverified", "No explicit conflict found; composition/certification is insufficient to confirm compatibility.", "; ".join(ambiguous), f"Ingredient exclusion: {key}")
    if field == "dietaryRestrictions" and key in NUTRITION_LABELS:
        return qualitative_nutrition(recipe_lookup[str(recipe_id)], *NUTRITION_LABELS[key])
    return verdict("unverified", "No justified compatibility rule for this diet or health condition.", value)

Check each persona against the recommended recipe

`persona_compatibility_df` records each selected restriction, allergy and condition, plus free-text notes requiring review. It checks the recommendation itself, separately from whether the explanation falsely claims compatibility. These rows must not be counted automatically as explanation hallucinations.

In [46]:
compatibility_rows = []
for rec in claim_index.to_dict("records"):
    profile = profile_lookup[rec["persona_id"]]["dietaryProfile"]
    for field in ["dietaryRestrictions", "foodAllergies", "healthConditions"]:
        for value in profile[field]["selected"]:
            compatibility_rows.append({**rec, "profile_field": field, "requirement": value,
                **check_profile_requirement(rec["recipe_id"], field, value)})
        note = profile[field]["other"].strip()
        if note:
            compatibility_rows.append({**rec, "profile_field": field + ".other", "requirement": note,
                **verdict("unverified", "Free-text persona requirement needs review.", note)})
persona_compatibility_df = pd.DataFrame(compatibility_rows)
display(pd.crosstab(persona_compatibility_df["profile_field"], persona_compatibility_df["verification_status"]))
display(persona_compatibility_df.loc[persona_compatibility_df["verification_status"].eq("contradicted")].head(10))

verification_status,contradicted,supported,unverified
profile_field,,,
dietaryRestrictions,48,53,365
dietaryRestrictions.other,0,0,536
foodAllergies,8,0,162
foodAllergies.other,0,0,66
healthConditions,0,0,414
healthConditions.other,0,0,152


,recommendation_id,persona_id,recipe_id,rank,profile_field,requirement,verification_status,reason,evidence,rule
12,user_01_rank_07,user_01,17107,7,dietaryRestrictions,Vegan,contradicted,Explicit source ingredient conflicts with this...,bacon,Ingredient exclusion: vegan
117,user_04_rank_03,user_04,73051,3,dietaryRestrictions,Vegetarian,contradicted,Explicit source ingredient conflicts with this...,slices bacon,Ingredient exclusion: vegetarian
228,user_06_rank_08,user_06,105281,8,dietaryRestrictions,High Protein,contradicted,Compared with the documented threshold.,Protein energy share: 12.00%,>= 20% energy from protein
232,user_06_rank_09,user_06,26470,9,dietaryRestrictions,High Protein,contradicted,Compared with the documented threshold.,Protein energy share: 11.64%,>= 20% energy from protein
236,user_06_rank_10,user_06,26440,10,dietaryRestrictions,High Protein,contradicted,Compared with the documented threshold.,Protein energy share: 11.22%,>= 20% energy from protein
240,user_06_rank_11,user_06,21623,11,dietaryRestrictions,High Protein,contradicted,Compared with the documented threshold.,Protein energy share: 10.29%,>= 20% energy from protein
244,user_06_rank_12,user_06,27130,12,dietaryRestrictions,High Protein,contradicted,Compared with the documented threshold.,Protein energy share: 12.23%,>= 20% energy from protein
268,user_07_rank_04,user_07,30469,4,dietaryRestrictions,Pescatarian,contradicted,Explicit source ingredient conflicts with this...,chicken bouillon cube,Ingredient exclusion: pescatarian
468,user_11_rank_04,user_11,138356,4,dietaryRestrictions,High Protein,contradicted,Compared with the documented threshold.,Protein energy share: 8.64%,>= 20% energy from protein
480,user_11_rank_08,user_11,20780,8,dietaryRestrictions,High Protein,contradicted,Compared with the documented threshold.,Protein energy share: 12.26%,>= 20% energy from protein


Verify extracted labels

A label's occurrence in the user's profile is recorded separately from recipe compatibility. Generic goals and medical-benefit claims cannot be validated by matching a profile label. Sentences acknowledging incompatibility or proposing modifications are sent for review so an honest warning is not automatically called false. Negated labels are also reviewed because 2.1 does not fully resolve negation scope.

In [47]:
compatibility_by_rec = {key: frame for key, frame in persona_compatibility_df.groupby("recommendation_id")}

def verify_label(row):
    profile = profile_lookup[row["persona_id"]]
    sentence = row["sentence_text"]
    key = label_key(row["term"])
    profile_values = [profile["dietary_goal"]] + [
        value for field in ["dietaryRestrictions", "foodAllergies", "healthConditions"]
        for value in profile["dietaryProfile"][field]["selected"]
    ]
    # Compare normalized words to the label produced by 2.1 (e.g. plural normalization).
    mentioned_profile = [value for value in profile_values if verification_words(value) == verification_words(row["term"])]
    checks = compatibility_by_rec.get(row["recommendation_id"])
    conflicts = [] if checks is None else checks.loc[checks["verification_status"].eq("contradicted"), "requirement"].tolist()
    extra = {"persona_label_match": bool(mentioned_profile), "matched_profile_values": "; ".join(mentioned_profile),
             "persona_conflicts": "; ".join(conflicts)}
    warning = re.search(r"\b(?:not|doesn't|does not|isn't|is not|doesn’t|isn’t)\b|\b(?:despite|although|however|caution)\b", sentence, re.I)
    if row["negated"] or warning or CONTEXT_REVIEW.search(sentence):
        return {**verdict("unverified", "Negation, warning or modification: review the intended compatibility claim.", sentence), **extra}
    if key in DIET_GROUPS:
        result = check_profile_requirement(row["recipe_id"], "dietaryRestrictions", key)
    elif key in NUTRITION_LABELS:
        # A user requirement alone does not assert that the recipe meets a nutrient threshold.
        if re.search(r"\b(?:align\w*|match\w*|fit\w*|meet\w*|support\w*|suitab\w*|ideal|perfect|complement\w*)\b", sentence, re.I):
            result = check_profile_requirement(row["recipe_id"], "dietaryRestrictions", key)
        else:
            result = verdict("unverified", "Profile nutrient label is mentioned without an explicit compatibility statement.", sentence)
    else:
        result = verdict("unverified", "Goal/condition or unsupported diet requires a specific evidence rule; profile agreement is not proof.", sentence)
    return {**result, **extra}

label_checks = verification_claims.loc[verification_claims["claim_type"].eq("label")].copy()
label_checks = label_checks.join(label_checks.apply(verify_label, axis=1).apply(pd.Series))
inspect_verification(label_checks)
display(label_checks[["text", "persona_label_match", "matched_profile_values", "persona_conflicts", "verification_status"]].head(10))

,claims
verification_status,
unverified,242
supported,6
contradicted,4


,claim_id,recipe_id,text,verification_status,reason,evidence
1299,1299,81968,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,Chickpeas With Spinach (Greek) is a fantastic ...
1300,1300,10614,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,"Refried Beans offers a high protein content, m..."
1301,1301,94673,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,Egyptian Red Lentil Soup is rich in protein an...
1302,1302,364252,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,"The olive oil adds healthy fats, making it a b..."
1303,1303,27778,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,It's a delightful way to enjoy black beans whi...
1304,1304,27778,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,It's a delightful way to enjoy black beans whi...
1305,1305,46221,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,Classic Black Beans and Rice is a traditional ...
1306,1306,12633,vegan,unverified,"Negation, warning or modification: review the ...",While it doesn't align perfectly with your veg...
1307,1307,46221,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,Classic Black Beans and Rice is a traditional ...
1308,1308,94673,muscle gain,unverified,Goal/condition or unsupported diet requires a ...,Egyptian Red Lentil Soup is rich in protein an...


,text,persona_label_match,matched_profile_values,persona_conflicts,verification_status
1299,muscle gain,True,Muscle Gain,,unverified
1300,muscle gain,True,Muscle Gain,,unverified
1301,muscle gain,True,Muscle Gain,,unverified
1302,muscle gain,True,Muscle Gain,,unverified
1303,muscle gain,True,Muscle Gain,,unverified
1304,muscle gain,True,Muscle Gain,,unverified
1305,muscle gain,True,Muscle Gain,,unverified
1306,vegan,True,Vegan,,unverified
1307,muscle gain,True,Muscle Gain,,unverified
1308,muscle gain,True,Muscle Gain,,unverified


Combine and save verification results

`verified_claims_df` keeps every extracted claim and its evidence. `persona_compatibility_df` is a separate recommendation/profile screen. `unverified` rows and automatic contradictions form the review queue. Counts below are claim-level results, not the explanation-level hallucination rate or a validation of the verifier.

In [48]:
verified_claims_df = pd.concat([ingredient_checks, nutrient_checks, label_checks], ignore_index=True).sort_values("claim_id")
verification_summary = pd.crosstab(verified_claims_df["claim_type"], verified_claims_df["verification_status"]).reindex(
    columns=["supported", "contradicted", "unverified"], fill_value=0
)
display(verification_summary)
review_claims_df = verified_claims_df.loc[verified_claims_df["verification_status"].ne("supported")].copy()

verified_claims_df.to_parquet(processed_dir / "verified_claims.parquet", index=False)
persona_compatibility_df.to_parquet(processed_dir / "persona_compatibility.parquet", index=False)
print("Saved verified_claims.parquet and persona_compatibility.parquet")
print("Claims for manual review:", len(review_claims_df))

verification_status,supported,contradicted,unverified
claim_type,,,
ingredient,332,53,119
label,6,4,242
nutrient,311,115,369


Saved verified_claims.parquet and persona_compatibility.parquet
Claims for manual review: 902


## 2.4 Validate the verifier itself

## 2.5 Hallucination rate and Hallucination patterns